# PokouAI — Fine-tune Gemma 4 E4B on Cocoa Diseases

**Goal**: QLoRA fine-tune Gemma 4 E4B (multimodal) on ~50k augmented cocoa pod images across 6 classes and 4 languages, export GGUF Q4_K_M for on-device inference.

**Platform**: Kaggle (T4 x2, 30 GB VRAM combined).

**Stack**: Unsloth (2× faster, ~60% less VRAM) + PEFT + TRL.

## Why Unsloth
Unsloth rewrites the Gemma forward/backward pass in Triton, enabling QLoRA fine-tuning of a 4B multimodal model on a single free-tier T4. Without it, QLoRA + gradient checkpointing still OOMs on the 16 GB T4 at sequence length 2048 with images. Unsloth makes this run at ~12 GB peak.

## 1 — Environment setup

In [ ]:
!pip install -qU "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -qU transformers accelerate peft trl datasets bitsandbytes pillow

In [ ]:
import os, json, torch
from pathlib import Path
from unsloth import FastModel
from trl import SFTTrainer, SFTConfig

print('CUDA:', torch.cuda.is_available(), '| devices:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')

## 2 — Load Gemma 4 E4B with 4-bit quantization

In [ ]:
MAX_SEQ_LEN = 2048
MODEL_NAME = 'unsloth/gemma-4-e4b-it-bnb-4bit'

model, tokenizer = FastModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    finetune_vision_layers=True,
    finetune_language_layers=True,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

## 3 — Load dataset

Expects the Kaggle dataset `pokou-ai-cocoa-training-data` mounted at `/kaggle/input/pokou-ai-cocoa-training-data/`.

In [ ]:
from datasets import load_dataset

DATA_ROOT = Path('/kaggle/input/pokou-ai-cocoa-training-data')
TRAIN_JSONL = DATA_ROOT / 'train.jsonl'
VAL_JSONL = DATA_ROOT / 'val.jsonl'

train_ds = load_dataset('json', data_files=str(TRAIN_JSONL), split='train')
val_ds = load_dataset('json', data_files=str(VAL_JSONL), split='train')

print('train:', len(train_ds), '| val:', len(val_ds))
print('first example keys:', train_ds[0].keys())

In [ ]:
# Format into Gemma multimodal chat template
from PIL import Image

def to_chat(example):
    messages = example['messages']
    user_content = messages[0]['content']
    assistant_content = messages[1]['content']
    image_path = next(c['path'] for c in user_content if c['type'] == 'image')
    user_text = next(c['text'] for c in user_content if c['type'] == 'text')
    assistant_text = assistant_content[0]['text']
    return {
        'image': Image.open(image_path).convert('RGB'),
        'conversations': [
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': user_text}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
        ],
    }

train_ds = train_ds.map(to_chat)
val_ds = val_ds.map(to_chat)

## 4 — Training config

In [ ]:
OUTPUT_DIR = '/kaggle/working/cocoa_v1'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        optim='adamw_8bit',
        weight_decay=0.01,
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=200,
        save_strategy='epoch',
        save_total_limit=3,
        bf16=True,
        max_seq_length=MAX_SEQ_LEN,
        dataset_text_field='conversations',
        report_to='none',
        seed=42,
    ),
)

## 5 — Train

In [ ]:
trainer_stats = trainer.train()
print(trainer_stats)

## 6 — Save LoRA adapter + merge to full weights

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('LoRA adapter saved to', OUTPUT_DIR)

## 7 — Export to GGUF Q4_K_M for on-device inference

Quantization target: 2.5-3.0 GB file, runs in ~4 GB RAM on llama.cpp.

In [ ]:
GGUF_OUTPUT = '/kaggle/working/cocoa_v1_gguf'
model.save_pretrained_gguf(GGUF_OUTPUT, tokenizer, quantization_method='q4_k_m')
!ls -lh /kaggle/working/cocoa_v1_gguf/

## 8 — Quick sanity check (single-image inference)

In [ ]:
from transformers import TextStreamer

sample = val_ds[0]
inputs = tokenizer.apply_chat_template(
    sample['conversations'][:1],
    add_generation_prompt=True,
    return_tensors='pt',
    tokenize=True,
).to('cuda')

streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(input_ids=inputs, max_new_tokens=400, streamer=streamer, do_sample=False)